# KuaiRand Data Understanding and EDA

This notebook uses PySpark/Spark SQL for scalable exploration. The intended flow is:

1. Inspect raw CSV files and schemas.
2. Convert CSV to Parquet in the bronze layer.
3. Use Parquet for row counts, missing values, duplicates, timestamps, interaction flags, and basic user/item/temporal statistics.
4. Prepare reusable columns for later session analysis and preference-drift analysis.

In [ ]:
from pathlib import Path
import os
import sys

os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")
os.environ.setdefault("PYSPARK_SUBMIT_ARGS", "--driver-memory 4g pyspark-shell")

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from pyspark.sql import functions as F
from pyspark.sql import Window

from recsys.data.kuairand import read_csv
from recsys.spark import get_spark

spark = get_spark("kuairand-eda")
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} | master={spark.sparkContext.master} | driver={spark.sparkContext.getConf().get('spark.driver.host')}")
spark


## Configure Paths

If your CSV files are still in `data/` during early local experiments, set `RAW_CSV_DIR = PROJECT_ROOT / "data"`. For the recommended layout, use `data/raw/kuairand`.

If you changed Spark memory or local IP settings after the first cell already ran, restart the kernel before continuing. Spark only applies those settings when the JVM starts.

In [ ]:
RAW_CSV_DIR = PROJECT_ROOT / "data"  # change to PROJECT_ROOT / "data/raw/kuairand" for the standard layout
BRONZE_PARQUET_DIR = PROJECT_ROOT / "data/bronze/kuairand"

USER_COL = "user_id"
ITEM_COL = "video_id"
DATE_COL = "date"
HOURMIN_COL = "hourmin"
TIME_MS_COL = "time_ms"

INTERACTION_COLS = [
    "is_click",
    "is_like",
    "is_follow",
    "is_comment",
    "is_forward",
    "is_hate",
    "long_view",
    "is_profile_enter",
]

## Inspect Raw Files

In [ ]:
csv_files = sorted(RAW_CSV_DIR.glob("*.csv"))
[(p.name, round(p.stat().st_size / 1024 / 1024, 3)) for p in csv_files]

In [ ]:
raw_tables = {}
for path in csv_files:
    name = path.stem
    df = read_csv(spark, path)
    raw_tables[name] = df
    print(f"\n{name}")
    df.printSchema()
    df.show(5, truncate=False)


## Convert CSV to Parquet

Parquet is the default format for analysis because it is columnar, typed, compressed, and much faster for repeated Spark scans than CSV.

In [ ]:
BRONZE_PARQUET_DIR.mkdir(parents=True, exist_ok=True)

for path in csv_files:
    name = path.stem
    target = BRONZE_PARQUET_DIR / name
    print(f"converting {path.name} -> {target}")
    df = read_csv(spark, path)
    df.write.mode("overwrite").parquet(str(target))
    print(f"wrote {target}")

spark.catalog.clearCache()


In [ ]:
parquet_tables = {}
for path in sorted(p for p in BRONZE_PARQUET_DIR.iterdir() if p.is_dir()):
    parquet_tables[path.name] = spark.read.parquet(str(path))

list(parquet_tables.keys())

## Pick Interaction Tables

KuaiRand has log tables plus user/video feature tables. This cell unions all log-like tables with matching columns for interaction-level analysis.

In [ ]:
log_names = [name for name, df in parquet_tables.items() if {USER_COL, ITEM_COL}.issubset(df.columns) and name.startswith("log_")]
log_names

In [ ]:
logs = None
for name in log_names:
    df = parquet_tables[name].withColumn("source_table", F.lit(name))
    logs = df if logs is None else logs.unionByName(df, allowMissingColumns=True)

logs.cache()
logs.printSchema()
logs.show(5, truncate=False)

## Row Counts, Users, Items

In [ ]:
table_counts = []
for name, df in parquet_tables.items():
    table_counts.append((name, df.count(), len(df.columns)))

spark.createDataFrame(table_counts, ["table", "rows", "columns"]).orderBy("table").show(truncate=False)

In [ ]:
logs.agg(
    F.count("*").alias("interactions"),
    F.countDistinct(USER_COL).alias("users"),
    F.countDistinct(ITEM_COL).alias("items"),
    F.countDistinct("source_table").alias("log_tables"),
).show(truncate=False)

## Missing Values and Duplicates

In [ ]:
def missing_summary(df):
    exprs = [F.sum(F.col(c).isNull().cast("long")).alias(c) for c in df.columns]
    return df.agg(*exprs)

missing_summary(logs).show(vertical=True, truncate=False)

In [ ]:
duplicate_keys = [USER_COL, ITEM_COL, DATE_COL, HOURMIN_COL, TIME_MS_COL]
available_duplicate_keys = [c for c in duplicate_keys if c in logs.columns]

logs.groupBy(*available_duplicate_keys).count().where(F.col("count") > 1).orderBy(F.desc("count")).show(20, truncate=False)

## Timestamp and Interaction Types

In [ ]:
def with_event_time(df):
    out = df
    if DATE_COL in out.columns:
        out = out.withColumn("event_date", F.to_date(F.col(DATE_COL).cast("string"), "yyyyMMdd"))
    if TIME_MS_COL in out.columns:
        out = out.withColumn("event_ts", F.to_timestamp(F.from_unixtime((F.col(TIME_MS_COL) / 1000).cast("long"))))
    if HOURMIN_COL in out.columns:
        out = out.withColumn("hourmin_str", F.lpad(F.col(HOURMIN_COL).cast("string"), 4, "0"))
        out = out.withColumn("event_hour", F.substring("hourmin_str", 1, 2).cast("int"))
        out = out.withColumn("event_minute", F.substring("hourmin_str", 3, 2).cast("int"))
    return out

logs_ts = with_event_time(logs).cache()
logs_ts.select(DATE_COL, HOURMIN_COL, TIME_MS_COL, "event_date", "event_ts", "event_hour", "event_minute").show(10, truncate=False)

In [ ]:
logs_ts.agg(
    F.min("event_date").alias("min_date"),
    F.max("event_date").alias("max_date"),
    F.min("event_ts").alias("min_ts"),
    F.max("event_ts").alias("max_ts"),
).show(truncate=False)

In [ ]:
existing_interaction_cols = [c for c in INTERACTION_COLS if c in logs_ts.columns]
interaction_summary = logs_ts.agg(*[F.sum(F.col(c).cast("long")).alias(c) for c in existing_interaction_cols])
interaction_summary.show(truncate=False)

## Basic User, Item, and Temporal Statistics

In [ ]:
user_stats = logs_ts.groupBy(USER_COL).agg(
    F.count("*").alias("interactions"),
    F.countDistinct(ITEM_COL).alias("distinct_items"),
    *[F.sum(F.col(c).cast("long")).alias(c) for c in existing_interaction_cols],
)

user_stats.orderBy(F.desc("interactions")).show(20, truncate=False)

In [ ]:
item_stats = logs_ts.groupBy(ITEM_COL).agg(
    F.count("*").alias("interactions"),
    F.countDistinct(USER_COL).alias("distinct_users"),
    *[F.sum(F.col(c).cast("long")).alias(c) for c in existing_interaction_cols],
)

item_stats.orderBy(F.desc("interactions")).show(20, truncate=False)

In [ ]:
logs_ts.groupBy("event_date").agg(
    F.count("*").alias("interactions"),
    F.countDistinct(USER_COL).alias("users"),
    F.countDistinct(ITEM_COL).alias("items"),
).orderBy("event_date").show(60, truncate=False)

In [ ]:
logs_ts.groupBy("event_hour").agg(
    F.count("*").alias("interactions"),
    F.countDistinct(USER_COL).alias("users"),
).orderBy("event_hour").show(24, truncate=False)

## Session and Preference-Drift Preparation

These columns are not final features yet. They prepare the interaction log for later sessionization and short-term preference-drift analysis.

In [ ]:
SESSION_GAP_MINUTES = 30

w_user_time = Window.partitionBy(USER_COL).orderBy("event_ts")

logs_sequence = (
    logs_ts
    .withColumn("prev_event_ts", F.lag("event_ts").over(w_user_time))
    .withColumn("gap_seconds", F.col("event_ts").cast("long") - F.col("prev_event_ts").cast("long"))
    .withColumn(
        "is_new_session",
        F.when(F.col("prev_event_ts").isNull(), F.lit(1))
        .when(F.col("gap_seconds") > SESSION_GAP_MINUTES * 60, F.lit(1))
        .otherwise(F.lit(0)),
    )
    .withColumn("session_index", F.sum("is_new_session").over(w_user_time.rowsBetween(Window.unboundedPreceding, 0)))
    .withColumn("session_id", F.concat_ws("_", F.col(USER_COL).cast("string"), F.col("session_index").cast("string")))
)

logs_sequence.select(USER_COL, ITEM_COL, "event_ts", "gap_seconds", "session_id", *existing_interaction_cols).show(20, truncate=False)

In [ ]:
session_stats = logs_sequence.groupBy("session_id", USER_COL).agg(
    F.min("event_ts").alias("session_start"),
    F.max("event_ts").alias("session_end"),
    F.count("*").alias("events"),
    F.countDistinct(ITEM_COL).alias("distinct_items"),
    *[F.sum(F.col(c).cast("long")).alias(c) for c in existing_interaction_cols],
)

session_stats.orderBy(F.desc("events")).show(20, truncate=False)

In [ ]:
daily_user_pref = logs_ts.groupBy(USER_COL, "event_date").agg(
    F.count("*").alias("interactions"),
    F.countDistinct(ITEM_COL).alias("distinct_items"),
    *[F.avg(F.col(c).cast("double")).alias(f"{c}_rate") for c in existing_interaction_cols],
)

daily_user_pref.orderBy(USER_COL, "event_date").show(20, truncate=False)